In [2]:
import pygame
import random
import sys
from pygame.locals import *

# 1. Initialize Pygame
pygame.init()

# Window dimensions
WINDOW_WIDTH = 800
WINDOW_HEIGHT = 600
CELL_SIZE = 20  # Size of each grid square

# Colors (RGB)
COLOR_BLACK = (0, 0, 0)
COLOR_WHITE = (255, 255, 255)
COLOR_GREEN = (0, 255, 0)       # Normal snake
COLOR_RED = (255, 0, 0)         # Regular fruit
COLOR_YELLOW = (255, 255, 0)     # Special fruit
COLOR_GREY = (100, 100, 100)     # Obstacles / Walls

# Set up display surface
screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
pygame.display.set_caption("AI Lab - Advanced Snake Game")

# Game Clock to control frame rate (speed)
clock = pygame.time.Clock()
BASE_SPEED = 10

# Fonts
font_style = pygame.font.SysFont("timesnewroman", 25)
game_over_font = pygame.font.SysFont("timesnewroman", 50)

# Helper function to display text on screen
def display_text(msg, color, x, y, center=False, large=False):
    font = game_over_font if large else font_style
    text_surface = font.render(msg, True, color)
    text_rect = text_surface.get_rect()
    if center:
        text_rect.center = (x, y)
    else:
        text_rect.topleft = (x, y)
    screen.blit(text_surface, text_rect)

# Main Game Loop Function
def game_loop():
    # Snake Initial Position & Body
    snake_x = 100
    snake_y = 100
    snake_body = [[snake_x, snake_y], [snake_x - CELL_SIZE, snake_y], [snake_x - (2 * CELL_SIZE), snake_y]]

    # Initial Movement Direction
    direction = "RIGHT"
    change_to = direction

    # Initial Game Conditions
    score = 0
    game_over = False

    # Food Setup (returns random grid-aligned coordinate)
    def generate_food_pos():
        x = random.randrange(1, (WINDOW_WIDTH // CELL_SIZE) - 1) * CELL_SIZE
        y = random.randrange(1, (WINDOW_HEIGHT // CELL_SIZE) - 1) * CELL_SIZE
        return [x, y]

    food_pos = generate_food_pos()
    yellow_food_pos = None  # Spawns only when score > 150

    # Defining Obstacles/Walls (Lab Task 2, Rule 3)
    # A simple horizontal wall barrier in the middle of the screen
    obstacles = [
        [300, 300], [320, 300], [340, 300], [360, 300], [380, 300], [400, 300],
        [420, 300], [440, 300], [460, 300], [480, 300], [500, 300]
    ]

    while not game_over:
        # --- 2. USER CONTROLS & EVENT HANDLING ---
        for event in pygame.event.get():
            if event.type == QUIT:
                pygame.quit()
                sys.exit()
            elif event.type == KEYDOWN:
                if event.key == K_UP and direction != "DOWN":
                    change_to = "UP"
                elif event.key == K_DOWN and direction != "UP":
                    change_to = "DOWN"
                elif event.key == K_LEFT and direction != "RIGHT":
                    change_to = "LEFT"
                elif event.key == K_RIGHT and direction != "LEFT":
                    change_to = "RIGHT"

        direction = change_to

        # Move the Snake Head
        if direction == "UP":
            snake_y -= CELL_SIZE
        elif direction == "DOWN":
            snake_y += CELL_SIZE
        elif direction == "LEFT":
            snake_x -= CELL_SIZE
        elif direction == "RIGHT":
            snake_x += CELL_SIZE

        # Growing mechanism: Insert new head position
        snake_body.insert(0, [snake_x, snake_y])

        # --- 3. SCORING MECHANISM & FRUIT EATING ---
        # Regular Red Fruit logic
        if snake_x == food_pos[0] and snake_y == food_pos[1]:
            score += 10
            food_pos = generate_food_pos()
        else:
            # Remove tail piece if no food eaten to simulate moving forward
            if not (yellow_food_pos and snake_x == yellow_food_pos[0] and snake_y == yellow_food_pos[1]):
                snake_body.pop()

        # Lab Task 2: Advanced Rules for Score > 150 (Yellow Fruit with Double Marks)
        if score > 150:
            if yellow_food_pos is None:
                yellow_food_pos = generate_food_pos()

            # Check if snake eats the Yellow Fruit
            if yellow_food_pos and snake_x == yellow_food_pos[0] and snake_y == yellow_food_pos[1]:
                score += 20  # Double marks
                yellow_food_pos = None  # Consumed, will respawn next loop
        else:
            yellow_food_pos = None  # Ensure it doesn't appear prematurely

        # --- 4. GAME OVER CONDITIONS ---
        # Collision with boundary walls
        if snake_x < 0 or snake_x >= WINDOW_WIDTH or snake_y < 0 or snake_y >= WINDOW_HEIGHT:
            game_over = True

        # Collision with its own self
        for block in snake_body[1:]:
            if snake_x == block[0] and snake_y == block[1]:
                game_over = True

        # Collision with static obstacles (Lab Task 2, Rule 3)
        for obstacle in obstacles:
            if snake_x == obstacle[0] and snake_y == obstacle[1]:
                game_over = True

        # --- 5. GRAPHICS RENDERING ---
        screen.fill(COLOR_BLACK)

        # Draw Obstacles
        for obstacle in obstacles:
            pygame.draw.rect(screen, COLOR_GREY, pygame.Rect(obstacle[0], obstacle[1], CELL_SIZE, CELL_SIZE))

        # Draw Snake
        for block in snake_body:
            pygame.draw.rect(screen, COLOR_GREEN, pygame.Rect(block[0], block[1], CELL_SIZE - 2, CELL_SIZE - 2))

        # Draw Regular Food (Red)
        pygame.draw.rect(screen, COLOR_RED, pygame.Rect(food_pos[0], food_pos[1], CELL_SIZE, CELL_SIZE))

        # Draw Special Food if available (Yellow)
        if yellow_food_pos:
            pygame.draw.rect(screen, COLOR_YELLOW, pygame.Rect(yellow_food_pos[0], yellow_food_pos[1], CELL_SIZE, CELL_SIZE))

        # Draw Scores & Instructions
        display_text(f"Score: {score}", COLOR_WHITE, 10, 10)

        # Lab Task 2: Speed modification notification
        current_speed = BASE_SPEED * 2 if score > 100 else BASE_SPEED
        if score > 100:
            display_text("SPEED: 2X ENGAGED!", COLOR_YELLOW, WINDOW_WIDTH - 220, 10)

        # Refresh Screen
        pygame.display.update()

        # Dynamic Game Speed management (Lab Task 2, Rule 1)
        clock.tick(current_speed)

    # --- 6. GAME OVER SCREEN ---
    screen.fill(COLOR_BLACK)
    display_text("GAME OVER", COLOR_RED, WINDOW_WIDTH // 2, WINDOW_HEIGHT // 2 - 50, center=True, large=True)
    display_text(f"Final Score: {score}", COLOR_WHITE, WINDOW_WIDTH // 2, WINDOW_HEIGHT // 2 + 10, center=True)
    display_text("Press R to Restart or Q to Quit", COLOR_GREY, WINDOW_WIDTH // 2, WINDOW_HEIGHT // 2 + 60, center=True)
    pygame.display.update()

    # Wait for choice to Restart or Close
    while True:
        for event in pygame.event.get():
            if event.type == QUIT:
                pygame.quit()
                sys.exit()
            if event.type == KEYDOWN:
                if event.key == K_q:
                    pygame.quit()
                    sys.exit()
                if event.key == K_r:
                    game_loop()  # Restart game recursively

# Run the game
if __name__ == "__main__":
    game_loop()

KeyboardInterrupt: 